# Feature Engineering

In this notebook, we shall identify the columns in our dataframe that will be of use to us when considering the default risk from new loans given. We will remove any columns that contain information unavailable at the point of generation of the loan. Such information will be of no use to us in the context of this model. We are only interested in what we can predict about the expected loss due to default at the beginning of the loan.

Let us first load our sample.

In [50]:
import pandas as pd

#load sample file from 01
df = pd.read_csv('data/sample.csv', index_col=0)

Let us again review the available columns in our dataset.

In [51]:
print(df.columns.tolist())

['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'acc_now_delinq',

### Key Information

There are of course some columns that contain key information regarding the details of the loan agreement. We will keep these in our dataframe for obvious reasons. This includes columns such as `loan_amnt`, `term` etc.; without these columns we could not later calculate expected losses. We also keep `id` to identify each loan, although this will not be used in our feature set.

For the other columns, it would be an unwise use of our time to justify the removal or non-removal of each column individually, due to the high amount. As such, we shall highlight key justifications only (including all of those that we shall keep).

The general principle for this removal exercise is that we shall remove all columns that contain information unavailable at the point of the loan's conception. This might include, for example, the removal of `hardship_amount`. This relates to hardship periods that began after the origination of the loan and was not information available at the time of conception. We also remove any columns that are simply of no use to us - for example `url`, which is irrelevant to the loan itself and is purely administrative.

We first discuss the columns that we shall keep in our dataframe as part of our feature set.

- We shall keep the abovementioned `loan amount` and `term` for obvious reasons, as these hold the objective details of the loan. Also in this category are `int_rate` and `installment`.
- `grade` and `sub_grade`: these are Lending Club risk grades. We keep these as they give an independent marker of borrower risk.
- `emp_length`, `home_ownership` and `annual_inc`: we keep these columns as they give an indication of the borrower's financial/employment situation. `verification_status` indicates whether income information was verified and so we keep this also.
- `purpose`: the reason for the loan - this is useful information as it may indicate that loans taken out for riskier purposes are more likely to default.
- `dti`: debt to income ratio. A high ratio in favour of debt indicates risk.
- `delinq_2yrs`: number of recent failures to pay an owed amount (a delinquency). Useful for reviewing recent reliability.
- `fico_range_low`, `inq_last_6mths`, `open_acc`, `total_acc`, `pub_rec` and `pub_rec_bankruptcies`: These are credit score metrics and information about the borrower's current credit arrangements (such as open/settled credit accounts and mortgages). FICO scores are the U.S. equivalent of credit scores and are one of the key metrics for borrower reliability. `pub_rec` are public derogatory records, i.e. public records of a failure to meet a credit obligation. Similarly, `pub_rec_bankruptcies` are bankruptcy records.
- `revol_bal` and `revol_util`: the amount and proportion (of total credit limit) of current credit, respectively. Useful for determining current liabilities.

We also give justifications for the non-inclusion of some specific columns.

- `fico_range_high`: instead of giving each borrower one FICO score, LendingClub recorded a range. As such, we take the lower value (assume credit score is worse) and disregard the upper value.
- Any columns related to settlements or payment plans. This are recorded after the origination of the loan.
- `addr_state` and `zip_code`: we are not considering regional risk in our model.
- `issue_d`: the issue date is not being considered in this model due to the historical nature of the dataset.

We now make the ammendment to our dataframe.

In [52]:
#list features from the columns as per the above, include default_flag from 01
features = ['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade',
    'emp_length', 'home_ownership', 'annual_inc', 'verification_status',
    'purpose', 'dti', 'delinq_2yrs', 'fico_range_low', 'inq_last_6mths',
    'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
    'mort_acc', 'pub_rec_bankruptcies', 'default_flag']

df = df[features]

#output to check
print(df.shape)
print(df.head())

(99996, 23)
         loan_amnt        term  int_rate  installment grade sub_grade  \
392949     32000.0   60 months     10.49       687.65     B        B3   
1273506     9600.0   36 months     12.99       323.42     C        C1   
324024      4000.0   36 months      6.68       122.93     A        A3   
2066630     6025.0   36 months     10.91       197.00     B        B4   
477199     25000.0   60 months     26.30       752.96     E        E5   

        emp_length home_ownership  annual_inc verification_status  ...  \
392949   10+ years       MORTGAGE    120000.0            Verified  ...   
1273506        NaN           RENT     21900.0            Verified  ...   
324024     4 years       MORTGAGE     83000.0        Not Verified  ...   
2066630  10+ years           RENT     52000.0        Not Verified  ...   
477199   10+ years            OWN     65000.0            Verified  ...   

        fico_range_low  inq_last_6mths  open_acc  pub_rec  revol_bal  \
392949           735.0          

### Missing Data

We now scan for missing data in our selected features.

In [53]:
#check missing values in selected features
missing = df.isnull().sum()
missing_pct = (missing / len(df)).round(3)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print(missing_df[missing_df['missing_count']>0].sort_values('missing_pct', ascending=False))

                      missing_count  missing_pct
emp_length                     6454        0.065
mort_acc                       2118        0.021
dti                              80        0.001
revol_util                       62        0.001
pub_rec_bankruptcies             48        0.000


The only features with a significant number of missing values are `emp_length` and `mort_acc`. For these features, we populate with the mode and median respectively. The mode is chosen for `emp_length` due to the nature of the data - it is given as a string representing categories (e.g. `4 years` or `10+ years`). So we choose the most common value.

We choose to remove all rows that are missing data across other features, due to the low volume. 

In [54]:
#fill emp_length with mode
df['emp_length'] = df['emp_length'].fillna(df['emp_length'].mode()[0])

#fill mort_acc with median
df['mort_acc'] = df['mort_acc'].fillna(df['mort_acc'].median())

# drop rows with missing dti, revol_util, pub_rec_bankruptcies
df = df.dropna(subset=['dti', 'revol_util', 'pub_rec_bankruptcies'])

# confirm no missing values remain
print(df.isnull().sum())
print(df.shape)

loan_amnt               0
term                    0
int_rate                0
installment             0
grade                   0
sub_grade               0
emp_length              0
home_ownership          0
annual_inc              0
verification_status     0
purpose                 0
dti                     0
delinq_2yrs             0
fico_range_low          0
inq_last_6mths          0
open_acc                0
pub_rec                 0
revol_bal               0
revol_util              0
total_acc               0
mort_acc                0
pub_rec_bankruptcies    0
default_flag            0
dtype: int64
(99807, 23)


## Data Types

We now convert the features without quantitive data into a data type we can work with.

In [55]:
#print the data types of each feature
print(df.dtypes)

loan_amnt               float64
term                     object
int_rate                float64
installment             float64
grade                    object
sub_grade                object
emp_length               object
home_ownership           object
annual_inc              float64
verification_status      object
purpose                  object
dti                     float64
delinq_2yrs             float64
fico_range_low          float64
inq_last_6mths          float64
open_acc                float64
pub_rec                 float64
revol_bal               float64
revol_util              float64
total_acc               float64
mort_acc                float64
pub_rec_bankruptcies    float64
default_flag              int64
dtype: object


We must encode the abovementioned object columns. First, we evaluate the unique values in each column.

In [56]:
#for loop to seach through the columns with object data type and output all of the unique values
for col in ['term', 'grade', 'sub_grade', 'emp_length', 
            'home_ownership', 'verification_status', 'purpose']:
    print(f'\n{col}:')
    print(df[col].value_counts())


term:
term
36 months    70912
60 months    28895
Name: count, dtype: int64

grade:
grade
B    29164
C    28873
A    18897
D    14407
E     6061
F     1847
G      558
Name: count, dtype: int64

sub_grade:
sub_grade
C1    6509
B5    6228
B4    6100
C2    5758
C4    5746
C3    5731
B3    5680
B2    5635
B1    5521
C5    5129
A5    4675
A4    4026
A1    3780
D1    3559
D2    3347
A3    3213
A2    3203
D3    2921
D4    2543
D5    2037
E1    1488
E2    1406
E3    1145
E4    1027
E5     995
F1     587
F2     413
F3     333
F4     279
F5     235
G1     175
G2     112
G3     104
G4      88
G5      79
Name: count, dtype: int64

emp_length:
emp_length
10+ years    39500
2 years       9009
< 1 year      8381
3 years       7976
1 year        6542
5 years       6238
4 years       6031
6 years       4446
7 years       4173
8 years       4011
9 years       3500
Name: count, dtype: int64

home_ownership:
home_ownership
MORTGAGE    48701
RENT        39804
OWN         11245
ANY            45
OTHER      

We work through each feature in turn.

### `term`
For `term`, the options are `36 months` and `60 months`. This is straightforward, we just extract the number of months. 

In [57]:
#extract number of months from term string
df['term'] = df['term'].str.extract(r'(\d+)').astype(int)
print(df['term'].value_counts())

term
36    70912
60    28895
Name: count, dtype: int64


### `grade` and `sub_grade`
Now, we must convert `grade` and `sub_grade`. We note that grades are of the form of a single letter whilst subgrades are paired with a number. As such, we can combine these columns to create a new column, where a number represents the subgrade in an ordered fashion. We first check that grade and subgrade align (i.e. that we have no instances wherein the grade is different to the letter component of the subgrade).

In [58]:
#extract the letter prefix from sub_grade
df['sub_grade_letter'] = df['sub_grade'].str[0]

#check they match grade
mismatch = df[df['grade'] != df['sub_grade_letter']]
print(f'Number of mismatches: {len(mismatch)}')

#remove temporary column
df = df.drop(columns=['sub_grade_letter'])

Number of mismatches: 0


We have that they are aligned.

Whilst it is reasonable to assume that grade `C1` is better than grade `C5` by convention, we perform an additional check to be sure. We check the default rate by subgrade, to see if those with lower numbered subgrades are less likely to default than those with higher number subgrades.



In [59]:
#calculate default rate by sub_grade to verify ordering
default_by_subgrade = df.groupby('sub_grade')['default_flag'].mean()
print(default_by_subgrade.sort_index())

sub_grade
A1    0.018519
A2    0.031221
A3    0.028322
A4    0.045206
A5    0.059679
B1    0.063032
B2    0.070807
B3    0.091021
B4    0.096721
B5    0.112075
C1    0.121063
C2    0.138416
C3    0.138719
C4    0.164462
C5    0.169429
D1    0.192751
D2    0.214819
D3    0.218418
D4    0.222572
D5    0.235150
E1    0.278226
E2    0.277383
E3    0.303930
E4    0.279455
E5    0.297487
F1    0.345826
F2    0.377724
F3    0.348348
F4    0.351254
F5    0.408511
G1    0.371429
G2    0.428571
G3    0.480769
G4    0.443182
G5    0.430380
Name: default_flag, dtype: float64


We see that subgrade `C1` is clearly stronger than subgrade `C5` and similarly for other grades. As such, we happily convert the `grade` and `sub_grade` columns to one ordered column.

In [60]:
#drop grade as sub_grade contains all the same information
df = df.drop(columns=['grade'])

#map sub_grade to ordered integer, A1=1 (lowest risk) through G5=35 (highest risk)
sub_grades = [f'{g}{n}' for g in 'ABCDEFG' for n in range(1, 6)]
sub_grade_map = {sg: i+1 for i, sg in enumerate(sub_grades)}
df['sub_grade'] = df['sub_grade'].map(sub_grade_map)

print(df['sub_grade'].value_counts().sort_index())

sub_grade
1     3780
2     3203
3     3213
4     4026
5     4675
6     5521
7     5635
8     5680
9     6100
10    6228
11    6509
12    5758
13    5731
14    5746
15    5129
16    3559
17    3347
18    2921
19    2543
20    2037
21    1488
22    1406
23    1145
24    1027
25     995
26     587
27     413
28     333
29     279
30     235
31     175
32     112
33     104
34      88
35      79
Name: count, dtype: int64


### `emp_length`

This one is very straightforward. We convert each string to just be the number of years in employment. For the edge cases: `< 1 year` we convert to `0` and `10+ years` we convert to `10`. Whilst the former is trivial, we can justify the latter in that job security remains meaningfully steady after ten years of employment. As such, the risk profile is similar for everyone in the `10+ years` category.

In [61]:
#emp_length: extract number and convert
df['emp_length'] = df['emp_length'].str.replace('< 1 year', '0')
df['emp_length'] = df['emp_length'].str.replace('10+ years', '10')
df['emp_length'] = df['emp_length'].str.extract(r'(\d+)').astype(float)

### `home_ownership`

For `home_ownership`, we first must consolidate our edge cases. We combine `ANY`, `NONE` and `OTHER` into one category (simply called `OTHER`) due to the negligible counts of `ANY` and `NONE`.

In [62]:
#consolidate edge cases
df['home_ownership'] = df['home_ownership'].replace(['ANY', 'NONE', 'OTHER'], 'OTHER')

Now, we shall convert this one column to three separate binary columns, representing a boolean value on whether the borrower owns, has a mortgage or rents. If all of these are set to `0` (false), then the borrower is in the `OTHER` category.

In [63]:
#encode, dropping OTHER as the reference category
df = pd.get_dummies(df, columns=['home_ownership'])
df = df.drop(columns=['home_ownership_OTHER'])

#confirm
print([col for col in df.columns if 'home_ownership' in col])

['home_ownership_MORTGAGE', 'home_ownership_OWN', 'home_ownership_RENT']


### `verification_status`

There are three possibilities here:
- `Not Verified`: borrower's income was not verified.
- `Verified`: borrower's income was verified.
- `Source Verified`: the source of the borrower's income (e.g. employer), is confirmed to exist.

We once again adopt a multi-boolean column approach, with `Verified` and `Source Verified` being the two columns.

In [64]:
#create boolean columns, dropping Not Verified as reference category
df = pd.get_dummies(df, columns=['verification_status'])
df = df.drop(columns=['verification_status_Not Verified'])

#confirm with output
print([col for col in df.columns if 'verification_status' in col])

['verification_status_Source Verified', 'verification_status_Verified']


### `purpose`

This is, perhaps, the most interesting of all categories. Our methodology for the treatment of the loan purpose is as follows:
1. Categorise each loan purpose by type.
1. Assign each type an inherent risk ranking: `1` (low risk), `2` (medium risk) or `3` (high risk).
1. Create a new column `purpose_risk`, containing the risk ranking for each loan.

So let us determine our categories.
- `1` (low risk):
    - `credit_card`: this is standard borrowing and does not indicate any underlying risk.
    - `car`, `major_purchase`, `wedding`, `home_improvement`, `moving`, `vacation`: these are one-off purchases or expenses, not necessarily indicative of any financial stress on the borrower.
- `2` (medium risk):
    - `debt_consolidation`: whilst consolidating one's loans is sensible financial behaviour, it also suggests a possibly tight or complicated financial situation.
    - `house`: a loan to assist in a house purchase suggests a slightly strained financial situation.
    - `medical`: an involuntary expense, often falling on the borrower in an unplanned manner.
- `3` (high risk):
    - `small_business`: businesses are naturally a volatile source of income. A small business is not as reliable as an employment income.
    - `renewable_energy` and `educational`: there is minimal data for these loans. This creates inherent uncertainty.
    - `other`: those with unknown purpose are assumed to be risky due to their non-standard nature.

Now we map these purposes to the risk.

In [65]:
#map purpose to risk tier based on above ranking
purpose_risk_map = {
    'credit_card': 1,
    'car': 1,
    'major_purchase': 1,
    'home_improvement': 1,
    'moving': 1,
    'vacation': 1,
    'wedding': 1,
    'debt_consolidation': 2,
    'house': 2,
    'medical': 2,
    'small_business': 3,
    'renewable_energy': 3,
    'educational': 3,
    'other': 3
}

df['purpose_risk'] = df['purpose'].map(purpose_risk_map)

#drop original purpose column
df = df.drop(columns=['purpose'])

#confirm
print(df['purpose_risk'].value_counts())
print(df['purpose_risk'].isnull().sum())

purpose_risk
2    58497
1    34069
3     7241
Name: count, dtype: int64
0


## Features

We convert the boolean columns to `int64` and then briefly confirm all columns are now numerical.

In [66]:
#conversion to int64 for ease of use
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

#confirmation that all columns are numerical
print(df.dtypes)
print(df.shape)

loan_amnt                              float64
term                                     int64
int_rate                               float64
installment                            float64
sub_grade                                int64
emp_length                             float64
annual_inc                             float64
dti                                    float64
delinq_2yrs                            float64
fico_range_low                         float64
inq_last_6mths                         float64
open_acc                               float64
pub_rec                                float64
revol_bal                              float64
revol_util                             float64
total_acc                              float64
mort_acc                               float64
pub_rec_bankruptcies                   float64
default_flag                             int64
home_ownership_MORTGAGE                  int64
home_ownership_OWN                       int64
home_ownershi

Now we save our features to a new file. We have processed the dataframe such that it is now in a form that shall be useful to us.

In [67]:
#save cleaned feature set for use in 03
df.to_csv('data/features.csv', index=True)